# 03 — Inference Strategies + ML Training with Lance

**Purpose:** Demonstrate three progressively scalable approaches to scene classification (weather, scene type, time of day) using the Lance `frames` dataset. The right choice depends on your scale and cost constraints — this notebook makes those trade-offs explicit.

---

## Why not just use a VLM for everything?

For well-understood visual categories like weather or scene type, modern vision-language models work zero-shot — no training required. The case for a trained classifier is almost entirely about **cost and throughput at scale**. At 10M frames, VLM API inference runs to tens of thousands of dollars and takes weeks. A trained classifier on stored CLIP embeddings costs pennies and takes hours.

This notebook shows all three approaches so you can choose based on your dataset size and latency requirements.

---

## Stage 1 — VLM Structured Output (LLaVA via Ray)

Run a vision-language model (LLaVA, served locally on Ray GPU workers) against a sample of frames. Prompt the model to return structured JSON with weather, scene type, and time-of-day fields. This approach produces the richest, most flexible output and requires no labeled training data — but is too expensive and slow to run against millions of frames.

**Use when:** Dataset is small (<100K frames), labels are ambiguous or open-ended, or you need to generate pseudo-labels for an unlabeled dataset.

## Stage 2 — CLIP Zero-Shot (uses stored embeddings, no training)

Use the CLIP embeddings already stored in Lance to classify frames via text-image cosine similarity. Encode each label as a natural language prompt (e.g., `"a dashcam photo taken in rainy weather"`) and find the closest match to each frame's stored embedding. No model training, no API calls — inference runs directly against the Lance dataset in milliseconds.

Benchmarks on BDD100K place CLIP zero-shot at ~70–75% accuracy for weather classification. For many production use cases this is sufficient, and the cost is effectively zero.

**Use when:** Label space is well-defined, accuracy requirements are moderate, and you want a fast production baseline with no training overhead.

## Stage 3 — CLIP Embeddings → Trained MLP Classifier

Train a lightweight MLP on top of frozen CLIP embeddings using BDD100K ground-truth labels (weather × scene × time-of-day). Load training batches via `lance.torch.data.LanceDataset` with column projection (`embedding` + labels only — no image bytes needed at training time). Three classification heads share the same embedding input.

Includes a Lance vs Delta data loading benchmark: time N random-access batches with Lance vs a shuffled Delta table at equivalent row counts. The gap widens past 10M rows where Delta's epoch-level shuffle becomes a distributed sort.

Log loss, per-head accuracy, and data loading throughput to MLflow. Pin the Lance dataset version as a run parameter for full reproducibility.

**Use when:** You need domain-adapted accuracy, deterministic inference, or throughput at scale (10M+ frames). Fine-tuning adapts to your specific camera hardware and data distribution in ways zero-shot cannot.

---

**Inputs:** Lance `frames` dataset v2 at `/Volumes/{catalog}/{schema}/{volume}/frames/` (with CLIP embeddings); BDD100K video-level labels joined to frames via `video_id`

**Outputs:** Stage 1 — structured VLM predictions on a sample; Stage 2 — zero-shot accuracy report; Stage 3 — trained classifier logged to MLflow + Lance vs Delta throughput benchmark